# Setup

## Load Dependencies

In [1]:
import os
import json
from dotenv import load_dotenv
from pymongo import MongoClient
import requests
import certifi

# Load environment variables
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

MONGODB_URI = os.getenv("MONGODB_URI")
print("MongoDB URI loaded:", "✅ Yes" if MONGODB_URI else "❌ Not found")

# Constants
DB_NAME = "tokoku_db"
MANUAL_COLLECTION = "manual_chunks"
PRODUCTS_COLLECTION = "products"

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
MANUAL_PATH = os.path.join(DATA_DIR, "cust_manual_knowledge_base.json")
PRODUCTS_PATH = os.path.join(DATA_DIR, "tokoku_products.json")

OLLAMA_URL = "http://localhost:11434/api/embeddings"
EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
EMBEDDING_DIM = 768

MongoDB URI loaded: ✅ Yes


## Connect to MongoDB Atlas

In [2]:
# Connect to MongoDB Atlas
client = MongoClient(MONGODB_URI, tlsCAFile=certifi.where())
db = client[DB_NAME]

manual_collection   = db[MANUAL_COLLECTION]
products_collection = db[PRODUCTS_COLLECTION]

try:
    client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas")
    print(f"   Database   : {DB_NAME}")
    print(f"   Collections: {MANUAL_COLLECTION}, {PRODUCTS_COLLECTION}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to MongoDB Atlas
   Database   : tokoku_db
   Collections: manual_chunks, products


## Load Knowledge Base

In [3]:
# Load knowledge base
with open(MANUAL_PATH, "r", encoding="utf-8") as f:
    manual_entries = json.load(f)

# Inspect manual
print(f"📘 Manual entries : {len(manual_entries)}")
print(f"   Fields         : {list(manual_entries[0].keys())}")
print(f"   Sample topic   : {manual_entries[0]['topic']}")
print(f"   Sample category: {manual_entries[0]['category']}")
print(f"   Tags type      : {type(manual_entries[0]['tags'])}")

📘 Manual entries : 62
   Fields         : ['topic', 'information', 'category', 'tags', 'section', 'subsection', 'page_number', 'source_file']
   Sample topic   : Tokoku Overview
   Sample category: Policy
   Tags type      : <class 'list'>


In [4]:
# Load products
with open(PRODUCTS_PATH, "r", encoding="utf-8") as f:
    products = json.load(f)

# Inspect products
print(f"🛍️  Products       : {len(products)}")
print(f"   Fields         : {list(products[0].keys())}")
print(f"   Sample title   : {products[0]['title']}")
print(f"   Sample price   : {products[0]['selling_price']}")
print(f"   discount_pct   : {products[0]['discount_pct']} ({type(products[0]['discount_pct']).__name__})")
print(f"   out_of_stock   : {products[0]['out_of_stock']} ({type(products[0]['out_of_stock']).__name__})")

🛍️  Products       : 659
   Fields         : ['_id', 'pid', 'title', 'brand', 'category', 'sub_category', 'original_sub_category_en', 'selling_price', 'selling_price_formatted', 'actual_price', 'actual_price_formatted', 'discount', 'discount_pct', 'average_rating', 'out_of_stock', 'description', 'product_details', 'seller', 'platform', 'currency']
   Sample title   : Printed Men Polo Neck Grey T-Shirt
   Sample price   : 57000
   discount_pct   : 70 (int)
   out_of_stock   : False (bool)


# Embedding & Ingestion

In [5]:
def build_embed_text(doc: dict, doc_type: str) -> str:
    """Build the text to embed based on document type.
    
    doc_type: "manual" or "product"
    """
    if doc_type == "manual":
        parts = [
            doc.get("topic", ""),
            doc.get("section", ""),
            doc.get("information", "")
        ]

    elif doc_type == "product":
        parts = [
            doc.get("title", ""),
            doc.get("brand", ""),
            doc.get("sub_category", ""),
            doc.get("description", "")
        ]

        # selectively include semantically rich detail fields
        USEFUL_DETAIL_KEYS = {
            "Fabric", "Type", "Suitable For", "Ideal For",
            "Fit", "Sleeve", "Pattern", "Closure",
            "Warna", "Bahan"
        }
        for detail in doc.get("product_details", []):
            for k, v in detail.items():
                if k in USEFUL_DETAIL_KEYS:
                    parts.append(f"{k}: {v}")
    else:
        raise ValueError(f"Unknown doc_type: {doc_type}. Use 'manual' or 'product'")

    return " | ".join(filter(None, parts))

In [6]:
def get_embedding(text: str) -> list[float]:
    """Call Ollama to embed a text string.
    Returns a list of floats of length EMBEDDING_DIM.
    """
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": EMBEDDING_MODEL,
            "prompt": text
        }
    )
    response.raise_for_status()
    embedding = response.json()["embedding"]

    # verify dimension matches expected
    if len(embedding) != EMBEDDING_DIM:
        raise ValueError(
            f"Unexpected embedding dimension: {len(embedding)} "
            f"(expected {EMBEDDING_DIM})"
        )
    return embedding

In [7]:
from tqdm import tqdm
import time

def ingest_documents(
    documents: list[dict],
    collection,
    doc_type: str,
    batch_size: int = 100
) -> None:
    """Embed and ingest documents into MongoDB Atlas.
    
    Args:
        documents  : list of dicts to embed and insert
        collection : pymongo collection handle
        doc_type   : "manual" or "product"
        batch_size : number of docs to insert per insert_many call
    """
    batch = []
    failed = []
    inserted = 0

    for doc in tqdm(documents, desc=f"Ingesting {doc_type}", unit="doc"):
        try:
            # build embed text and get vector
            embed_text = build_embed_text(doc, doc_type)
            embedding = get_embedding(embed_text)

            # attach embedding to a copy of the doc
            doc_with_embedding = doc.copy()
            doc_with_embedding["embedding"] = embedding
            doc_with_embedding["embed_text"] = embed_text

            batch.append(doc_with_embedding)

            # flush batch to MongoDB when batch_size reached
            if len(batch) >= batch_size:
                collection.insert_many(batch)
                inserted += len(batch)
                batch = []

        except Exception as e:
            failed.append({
                "doc": doc.get("topic") or doc.get("title") or "unknown",
                "error": str(e)
            })
            continue

    # flush remaining docs in buffer
    if batch:
        collection.insert_many(batch)
        inserted += len(batch)

    #  Summary
    print(f"\n{'='*50}")
    print(f"✅ Inserted : {inserted} {doc_type} documents")
    print(f"❌ Failed   : {len(failed)} documents")
    if failed:
        print(f"\nFailed documents:")
        for f in failed:
            print(f"  - {f['doc']}: {f['error']}")
    print(f"{'='*50}")

In [8]:
# Safety check: drop collections before ingesting
print("🗑️  Dropping existing collections...")

manual_collection.drop()
products_collection.drop()

print(f"   {MANUAL_COLLECTION}   : dropped")
print(f"   {PRODUCTS_COLLECTION} : dropped")
print()

🗑️  Dropping existing collections...
   manual_chunks   : dropped
   products : dropped



In [9]:
# Ingest manual knowledge base
print("📘 Ingesting manual knowledge base...")
ingest_documents(
    documents=manual_entries,
    collection=manual_collection,
    doc_type="manual",
    batch_size=10       
)

📘 Ingesting manual knowledge base...


Ingesting manual:   0%|          | 0/62 [00:00<?, ?doc/s]

Ingesting manual: 100%|██████████| 62/62 [02:19<00:00,  2.25s/doc]


✅ Inserted : 62 manual documents
❌ Failed   : 0 documents


In [10]:
# Ingest products
print("🛍️  Ingesting products...")
ingest_documents(
    documents=products,
    collection=products_collection,
    doc_type="product",
    batch_size=100
)

🛍️  Ingesting products...


Ingesting product: 100%|██████████| 659/659 [24:10<00:00,  2.20s/doc]



✅ Inserted : 659 product documents
❌ Failed   : 0 documents


In [20]:
# Verify final counts in MongoDB
manual_count   = manual_collection.count_documents({})
products_count = products_collection.count_documents({})

# verify embedding field exists on all docs
manual_with_embedding   = manual_collection.count_documents({"embedding": {"$exists": True}})
products_with_embedding = products_collection.count_documents({"embedding": {"$exists": True}})

print(f"📘 manual_chunks")
print(f"   Total documents    : {manual_count}")
print(f"   With embedding     : {manual_with_embedding}")
print(f"   Missing embedding  : {manual_count - manual_with_embedding}")

print()

print(f"🛍️  products")
print(f"   Total documents    : {products_count}")
print(f"   With embedding     : {products_with_embedding}")
print(f"   Missing embedding  : {products_count - products_with_embedding}")

print()

all_good = (
    manual_count   == manual_with_embedding  == 62 and
    products_count == products_with_embedding == 659
)

print("✅ All documents ingested with embeddings" if all_good else "⚠️ Some documents missing — check above")

📘 manual_chunks
   Total documents    : 62
   With embedding     : 62
   Missing embedding  : 0

🛍️  products
   Total documents    : 659
   With embedding     : 659
   Missing embedding  : 0

✅ All documents ingested with embeddings


In [12]:
MANUAL_INDEX_NAME   = "manual_vector_index"

manual_index_definition = {
    "name": MANUAL_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": EMBEDDING_DIM,
                "similarity": "cosine"
            }
        ]
    }
}

In [13]:
PRODUCTS_INDEX_NAME = "product_vector_index"

products_index_definition = {
    "name": PRODUCTS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": EMBEDDING_DIM,
                "similarity": "cosine"
            },
            # filter fields — must be declared here to be used
            # inside $vectorSearch filter parameter
            {
                "type": "filter",
                "path": "selling_price"
            },
            {
                "type": "filter",
                "path": "discount_pct"
            },
            {
                "type": "filter",
                "path": "sub_category"
            },
            {
                "type": "filter",
                "path": "out_of_stock"
            }
        ]
    }
}

In [14]:
def create_vector_index(collection, index_definition: dict) -> None:
    index_name = index_definition["name"]
    try:
        collection.create_search_index(index_definition)
        print(f"✅ Index created   : {index_name}")
        print(f"   Collection      : {collection.name}")
        print(f"   Status          : building (takes ~1-2 minutes in Atlas)")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"⚠️  Index already exists: {index_name} — skipping")
        else:
            print(f"❌ Failed to create index: {index_name}")
            print(f"   Error: {e}")

In [15]:
print("Creating vector search indexes...")
print()
create_vector_index(manual_collection, manual_index_definition)
print()
create_vector_index(products_collection, products_index_definition)
print()
print("⏳ Indexes are building in Atlas.")
print("   Wait 1-2 minutes before running any $vectorSearch queries.")
print("   You can monitor progress in Atlas UI:")
print("   Database → Collections → Search Indexes tab")

Creating vector search indexes...

✅ Index created   : manual_vector_index
   Collection      : manual_chunks
   Status          : building (takes ~1-2 minutes in Atlas)

✅ Index created   : product_vector_index
   Collection      : products
   Status          : building (takes ~1-2 minutes in Atlas)

⏳ Indexes are building in Atlas.
   Wait 1-2 minutes before running any $vectorSearch queries.
   You can monitor progress in Atlas UI:
   Database → Collections → Search Indexes tab


# Verification

In [16]:
def test_vector_search(collection, index_name: str, query: str, doc_type: str) -> None:
    query_embedding = get_embedding(query)

    pipeline = [
        {
            "$vectorSearch": {
                "index": index_name,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 20,
                "limit": 3
            }
        },
        {
            "$project": {
                "embedding": 0,      # exclude — too large to print
                "embed_text": 0,     # exclude — already know it works
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]

    results = list(collection.aggregate(pipeline))

    print(f"Query : '{query}'")
    print(f"Index : {index_name}")
    print(f"Results: {len(results)}")
    print()

    for i, r in enumerate(results):
        score = r.get("score", 0)
        if doc_type == "manual":
            print(f"  [{i+1}] score={score:.3f} | {r['topic']} | {r['section']}")
        else:
            print(f"  [{i+1}] score={score:.3f} | {r['title']} | {r['selling_price_formatted']}")
    print()

In [21]:
# Test manual index
print("📘 Testing manual_vector_index...")
test_vector_search(
    collection=manual_collection,
    index_name=MANUAL_INDEX_NAME,
    query="how do I return a product?",
    doc_type="manual"
)

📘 Testing manual_vector_index...
Query : 'how do I return a product?'
Index : manual_vector_index
Results: 3

  [1] score=0.769 | Return Policy | 7. Returns, Refunds & Disputes
  [2] score=0.763 | Filing a Return/Complaint | 7. Returns, Refunds & Disputes
  [3] score=0.731 | Seller Return Refusal | 11. Frequently Asked Questions (FAQ)



In [22]:
# Test product index
print("🛍️  Testing product_vector_index...")
test_vector_search(
    collection=products_collection,
    index_name=PRODUCTS_INDEX_NAME,
    query="warm jacket for cold weather",
    doc_type="product"
)

🛍️  Testing product_vector_index...
Query : 'warm jacket for cold weather'
Index : product_vector_index
Results: 3

  [1] score=0.746 | Full Sleeve Striped Men Casual Jacket | Rp 133.000
  [2] score=0.729 | Sleeveless Solid Men Padded Jacket | Rp 733.000
  [3] score=0.718 | Graphic Print Winter Men Gloves | Rp 44.000



In [23]:
# Test hybrid filter on product index
print("🔍 Testing hybrid filter (jacket under Rp 200.000)...")
query_embedding = get_embedding("warm jacket")
pipeline = [
    {
        "$vectorSearch": {
            "index": PRODUCTS_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 100,
            "limit": 3,
            "filter": {"selling_price": {"$lte": 200000}}
        }
    },
    {
        "$project": {
            "title": 1,
            "selling_price_formatted": 1,
            "sub_category": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }
]
results = list(products_collection.aggregate(pipeline))
print(f"Query : 'warm jacket' with filter price ≤ Rp 200.000")
print(f"Results: {len(results)}")
for i, r in enumerate(results):
    print(f"  [{i+1}] score={r['score']:.3f} | {r['title']} | {r['selling_price_formatted']}")

🔍 Testing hybrid filter (jacket under Rp 200.000)...
Query : 'warm jacket' with filter price ≤ Rp 200.000
Results: 3
  [1] score=0.713 | Full Sleeve Striped Men Casual Jacket | Rp 133.000
  [2] score=0.703 | Full Sleeve Printed Men Casual Jacket | Rp 190.000
  [3] score=0.701 | Welwear suncool1009 Nylon Arm Warmer  (Multicolor) | Rp 28.000
